[← Previous: Introduction](../main.ipynb)

# Data preparation

## The MFCAD dataset

MFCAD is a **synthetic CAD dataset** introduced by Cao et al. [[3]](https://github.com/hducg/MFCAD). Its solids are generated by applying machining features to a stock cube, with multiple features in a solid. These are procedurally constructed parts rather than scans or a collection of industrial designs. The released benchmark contains 15,488 solids with planar faces and 16 face-label categories, including the remaining stock material [[1]](https://openaccess.thecvf.com/content/CVPR2021/papers/Jayaraman_UV-Net_Learning_From_Boundary_Representations_CVPR_2021_paper.pdf), [[4]](https://doi.org/10.1016/j.cad.2022.103226).

### Boundary representation (B-rep)

MFCAD represents each solid using **boundary representation (B-rep)**, which describes a solid through its bounding faces, edges, and vertices. A face is a bounded region of a parametric surface, an edge is a bounded segment of a parametric curve, and vertices mark edge endpoints. Boundary loops trim surfaces to define the visible faces, including any holes.

B-rep stores both **geometry**—the parametric surfaces, curves, and point coordinates—and **topology**—how faces, edges, and vertices connect. Face adjacency is therefore already available through shared B-rep edges. We view this connectivity as a face-adjacency graph: nodes represent faces, and graph edges connect faces sharing a B-rep edge. Each node and edge is associated with its existing surface or curve geometry.

![B-rep entities and their face-adjacency graph](../figures/uvnet-brep.png)

*B-rep entities and the corresponding face-adjacency graph. Reproduced from Figure 2(a–b) of [Jayaraman et al. [1]](https://openaccess.thecvf.com/content/CVPR2021/papers/Jayaraman_UV-Net_Learning_From_Boundary_Representations_CVPR_2021_paper.pdf).*

## UV-Net: sampling B-rep geometry

UV-Net builds on this B-rep representation by sampling the parametric surfaces and curves into regular grids [[1]](https://openaccess.thecvf.com/content/CVPR2021/papers/Jayaraman_UV-Net_Learning_From_Boundary_Representations_CVPR_2021_paper.pdf). This provides a practical way to **standardize varied parametric geometry into a common input format for machine learning**. Different surface and curve types can have different mathematical descriptions, but evaluating them on fixed-size parameter grids produces tensors with consistent dimensions and channels. These tensors can be processed by shared CNNs while retaining local geometric structure. The grids standardize the input format; they do not guarantee invariance to different parameterizations of the same geometry.

![Sampled curve and surface grids](../figures/uvnet-sampling.png)

*Regular parameter grids for curve and surface geometry. Reproduced from Figure 2(c) of Jayaraman et al. [[1]](https://openaccess.thecvf.com/content/CVPR2021/papers/Jayaraman_UV-Net_Learning_From_Boundary_Representations_CVPR_2021_paper.pdf).*

### Face data

In the B-rep, each face lies on a parametric surface $S(u,v)\in\mathbb{R}^3$: the surface parameters $u$ and $v$ locate a point on that surface. UV-Net uses this existing parameterization to sample a regular grid over the bounds enclosing the trimmed face. In UV-Net’s prepared release, each face has **10 × 10 samples**, with seven channels per sample:

| Channels | Meaning |
| --- | --- |
| 0–2 | 3D position $(x,y,z)$ |
| 3–5 | Unit surface normal $(n_x,n_y,n_z)$ |
| 6 | Binary trimming mask: 1 for samples on or inside the face boundary, 0 outside |

The parameter values are uniformly spaced, including both endpoints:

$$
u_i=u_{\min}+\frac{i}{9}(u_{\max}-u_{\min}),\qquad
v_j=v_{\min}+\frac{j}{9}(v_{\max}-v_{\min}),\qquad i,j=0,\ldots,9.
$$

The rectangular parameter grid can extend beyond the visible face or cross holes. The trimming mask records which samples belong to the face. Uniform spacing in parameter space does not generally imply uniform spacing or equal-area coverage on the 3D surface. The surface normal describes local orientation, while the grid preserves local neighborhoods for a 2D CNN.

### Edge data

Similarly, each B-rep edge lies on a parametric curve $C(t)\in\mathbb{R}^3$, where $t$ locates a point along the curve. UV-Net samples this existing curve parameterization. Each graph edge carries **10 curve samples**, uniformly spaced over its bounded parameter interval using the same endpoint-inclusive rule. Each sample has six channels: three position coordinates and three components of the **unit curve tangent**. The tangent follows the curve direction; it is distinct from the surface normal stored for a face. These 1D grids can be processed by a 1D CNN [2].

### Stored tensors and labels

For a solid with $N$ graph nodes and $E$ graph edges, `graph.ndata["x"]` has shape `(N, 10, 10, 7)` and `graph.edata["x"]` has shape `(E, 10, 6)`. Edge rows follow the stored graph-edge order. Face annotations provide the machining-feature class for each node.

## Prediction task

The task is to predict a machining-feature label for **each face**, rather than one label for an entire solid. Several faces can belong to the same feature. This makes the relationship between a face and its neighbors relevant: a locally similar planar face can play different roles in different feature structures.

## Prepared data used in this project

We use the **prepared MFCAD data released by the UV-Net authors**, avoiding the need to regenerate the graphs, sample the geometry, and assemble the face labels [[5]](https://github.com/AutodeskAILab/UV-Net#MFCAD). The CAD solids and face annotations originate from MFCAD; the face-adjacency graphs and sampled geometry are supplied in UV-Net's processed release.

Our extraction and loading code reads these existing graphs, annotations, and predefined splits; it does not regenerate the solids, resample their geometry, or perform a new cleaning procedure. We retain the supplied training, validation, and test splits for comparison across models. Results therefore characterize this synthetic benchmark; generalization to industrial CAD parts requires separate evaluation.


### Checking class names

Unexpected class–geometry relationships in the initial EDA prompted a manual review. We visually inspected two examples per machining-feature class and corrected the class-name mismatches based on their geometry. The [mapping](../data/label_mapping.json) and [review log](../working/label_review.json) are included; the raw annotations and numeric class IDs remain unchanged.

We unpack the supplied archive following the [dataset extraction instructions](../README.md#dataset-extraction).

In [1]:
%run scripts/data.py
unpack_data()

Dataset ready: data/MFCADDataset
Graphs: 15,488
Label files: 15,488


## Loading the data

We load the three predefined splits. Each sample contains a face-adjacency graph and its corresponding face annotations.

In [2]:
%run scripts/data.py

In [3]:
train_data = load_mfcad("train")
val_data = load_mfcad("validation")
test_data = load_mfcad("test")

print(f"Training: {len(train_data):,} solids")
print(f"Validation: {len(val_data):,} solids")
print(f"Test: {len(test_data):,} solids")

Training: 9,277 solids
Validation: 3,090 solids
Test: 3,094 solids


## References

[[1]](https://openaccess.thecvf.com/content/CVPR2021/papers/Jayaraman_UV-Net_Learning_From_Boundary_Representations_CVPR_2021_paper.pdf) Jayaraman, P. K., et al. (2021). [UV-Net: Learning From Boundary Representations](https://openaccess.thecvf.com/content/CVPR2021/papers/Jayaraman_UV-Net_Learning_From_Boundary_Representations_CVPR_2021_paper.pdf). CVPR, 11703–11712. Section 3.1 and Figure 2.

[2] Autodesk AI Lab. [UV-Net preprocessing implementation](https://github.com/AutodeskAILab/UV-Net/blob/main/process/solid_to_graph.py). Face and curve sampling and channel construction.

[[3]](https://github.com/hducg/MFCAD) Cao, W., et al. (2020). *Graph Representation of 3D CAD Models for Machining Feature Recognition With Deep Learning*. ASME IDETC-CIE. Original MFCAD dataset repository.

[[4]](https://doi.org/10.1016/j.cad.2022.103226) Colligan, A. R., et al. (2022). *Hierarchical CADNet: Learning from B-Reps for Machining Feature Recognition*. Computer-Aided Design, 147, 103226. Section 6.2 describes MFCAD's construction and planar features.

[[5]](https://github.com/AutodeskAILab/UV-Net#MFCAD) Autodesk AI Lab. *UV-Net: MFCAD data release*. Source of the prepared dataset used in this report.


---

[← Previous: Introduction](../main.ipynb) | [Next: Exploratory data analysis →](02_eda.ipynb)